In [6]:
import json
import re
import time
import numpy as np
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from typing import Dict, Any, List, Set, Tuple, Optional

from SPARQLWrapper import SPARQLWrapper, JSON as SPARQL_JSON
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize as sk_normalize
from src.embedder import SPARQLEmbedder


SEMANTIC_PATH = "../dataset/raw/jaccard_triplets_dbpedia.json" 
SPLITS_PATH = "../dataset/splits/splits_dbpedia.json"
CHECKPOINT_PATH = "../checkpoints/dbpedia_model_semantic-20.pt"

QWEN_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
MINILM_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
E5_MODEL_NAME = "intfloat/e5-base-v2"
MPNET_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

RESULT_LIMIT = 200
TOPKS = [3, 5, 10, 20]
BATCH = 32
JACCARD_BINARY_THRESHOLD = 0.8

SPARQL_SLEEP_SECONDS = 0.0

with open(SEMANTIC_PATH, "r", encoding="utf-8") as f:
    raw_sem = json.load(f)

with open(SPLITS_PATH, "r", encoding="utf-8") as f:
    split_info = json.load(f)

test_idx = split_info.get("test_idx", split_info.get("valid_idx"))
print(f"Loaded {len(raw_sem)} semantic entries | Eval size: {len(test_idx)}")


endpoint = SPARQLWrapper("https://dbpedia.data.dice-research.org/sparql")
endpoint.setReturnFormat(SPARQL_JSON)


# -------------------------------------------------------------------
# Query execution cache and bookkeeping
# -------------------------------------------------------------------
# answers are stored as tuple rows to avoid string collision issues
answer_cache: Dict[str, Set[Tuple[str, ...]]] = {}
query_status: Dict[str, str] = {}   # "ok", "empty", "failed"


def normalize_query(q: str) -> str:
    return "\n".join(line.strip() for line in q.strip().splitlines())


def add_limit_if_missing(query: str, limit: int) -> str:
    q = query.strip().rstrip(";")
    if re.search(r"\bLIMIT\b", q, flags=re.IGNORECASE):
        return q
    return f"{q}\nLIMIT {limit}"


def get_answers(query: str, limit: int = 200, verbose: bool = False) -> Set[Tuple[str, ...]]:
    q_norm = normalize_query(query)
    if q_norm in answer_cache:
        return answer_cache[q_norm]

    q_exec = add_limit_if_missing(q_norm, limit)
    endpoint.setQuery(q_exec)

    try:
        res = endpoint.query().convert()
        bindings = res.get("results", {}).get("bindings", [])
        vars_ = res.get("head", {}).get("vars", [])

        answers: Set[Tuple[str, ...]] = set()
        for b in bindings:
            vals = []
            for v in vars_:
                if v in b:
                    vals.append(b[v]["value"])
            if vals:
                answers.add(tuple(vals))

        answer_cache[q_norm] = answers
        query_status[q_norm] = "empty" if len(answers) == 0 else "ok"

    except Exception as e:
        if verbose:
            print(f"[SPARQL ERROR] {e}\n--- Query ---\n{q_exec}\n-------------")
        answer_cache[q_norm] = set()
        query_status[q_norm] = "failed"

    if SPARQL_SLEEP_SECONDS > 0:
        time.sleep(SPARQL_SLEEP_SECONDS)

    return answer_cache[q_norm]


def jaccard(a: Set[Tuple[str, ...]], b: Set[Tuple[str, ...]]) -> float:
    # safer version:
    # empty-empty should not count as a perfect semantic match
    if not a and not b:
        return 0.0
    if not a or not b:
        return 0.0
    inter = len(a & b)
    union = len(a | b)
    return inter / union if union else 0.0


def safe_jaccard_for_queries(q1: str, q2: str) -> Optional[float]:
    q1n = normalize_query(q1)
    q2n = normalize_query(q2)

    s1 = query_status.get(q1n)
    s2 = query_status.get(q2n)

    # If either query failed, skip graded comparison entirely
    if s1 == "failed" or s2 == "failed":
        return None

    a = answer_cache.get(q1n, set())
    b = answer_cache.get(q2n, set())
    return jaccard(a, b)


def get_anchor_query(item: Dict[str, Any]) -> str:
    return item.get("anchor_query") or item.get("base_query") or ""


def build_pool_queries_for_item(item: Dict[str, Any], positives_only: bool = True) -> List[str]:
    pool: List[str] = []

    for p in item.get("positive_queries", []):
        q = p.get("query") if isinstance(p, dict) else p
        if q:
            pool.append(q)

    if not positives_only:
        for n in item.get("negative_queries", []):
            q = n.get("query") if isinstance(n, dict) else n
            if q:
                pool.append(q)

    seen = set()
    out = []
    for q in pool:
        if q not in seen:
            out.append(q)
            seen.add(q)
    return out


# -------------------------------------------------------------------
# Build the unique query universe first
# -------------------------------------------------------------------
unique_queries: List[str] = []
seen_q: Set[str] = set()


def add_unique(q: str):
    if q and q not in seen_q:
        seen_q.add(q)
        unique_queries.append(q)


for idx in test_idx:
    item = raw_sem[idx]
    anchor_q = get_anchor_query(item)
    if not anchor_q:
        continue

    add_unique(anchor_q)

    # include both positives and negatives so the corpus is broad
    for q in build_pool_queries_for_item(item, positives_only=False):
        add_unique(q)

print(f"Unique queries to execute/encode: {len(unique_queries)}")


# -------------------------------------------------------------------
# Full-corpus retrieval evaluation setup
# Each anchor retrieves over the whole query corpus except itself
# Positives = annotated positive queries only
# -------------------------------------------------------------------
global_corpus = list(unique_queries)

eval_items_corrected: List[Tuple[str, List[str], Set[str]]] = []
for idx in test_idx:
    item = raw_sem[idx]
    anchor_q = get_anchor_query(item)
    if not anchor_q:
        continue
    if anchor_q not in seen_q:
        continue

    pool = [q for q in global_corpus if q != anchor_q]

    positives: Set[str] = set()
    for p in item.get("positive_queries", []):
        q = p.get("query") if isinstance(p, dict) else p
        if q and q in seen_q and q != anchor_q:
            positives.add(q)

    if not positives:
        continue

    eval_items_corrected.append((anchor_q, pool, positives))

print(f"Corrected eval items with non-empty annotated positives: {len(eval_items_corrected)}")


# -------------------------------------------------------------------
# Execute SPARQL and cache results
# -------------------------------------------------------------------
print("Executing SPARQL for unique queries (cached)...")
for q in unique_queries:
    _ = get_answers(q, limit=RESULT_LIMIT, verbose=False)

n_ok = sum(1 for s in query_status.values() if s == "ok")
n_empty = sum(1 for s in query_status.values() if s == "empty")
n_failed = sum(1 for s in query_status.values() if s == "failed")
print(f"Cached answers: {len(answer_cache)} | ok: {n_ok} | empty: {n_empty} | failed: {n_failed}")


# -------------------------------------------------------------------
# Optional graded relevance using Jaccard over full corpus
# This is more expensive, but kept here if you want it.
# -------------------------------------------------------------------
jaccard_scores_per_item: List[Dict[str, float]] = []
graded_valid_mask: List[bool] = []

print("Building graded Jaccard relevance maps...")
for anchor_q, pool, _ in eval_items_corrected:
    rel: Dict[str, float] = {}
    valid_count = 0

    for q in pool:
        score = safe_jaccard_for_queries(anchor_q, q)
        if score is not None:
            rel[q] = score
            valid_count += 1

    jaccard_scores_per_item.append(rel)
    graded_valid_mask.append(valid_count > 0)

print(f"Items with at least one valid graded Jaccard comparison: {sum(graded_valid_mask)} / {len(eval_items_corrected)}")


# -------------------------------------------------------------------
# Embedding helpers
# -------------------------------------------------------------------
def build_emb_map(
    model,
    queries: List[str],
    batch_size: int = 32,
    transform=None,
    normalize_embeddings: bool = True,
    use_convert_to_numpy: bool = True
) -> Dict[str, np.ndarray]:
    emb_map: Dict[str, np.ndarray] = {}

    for i in range(0, len(queries), batch_size):
        batch_qs = queries[i:i + batch_size]
        texts = [transform(q) for q in batch_qs] if transform else batch_qs

        if use_convert_to_numpy:
            batch_embs = model.encode(
                texts,
                convert_to_numpy=True,
                normalize_embeddings=normalize_embeddings
            )
        else:
            batch_embs = model.encode(texts)
            if normalize_embeddings:
                batch_embs = batch_embs / (np.linalg.norm(batch_embs, axis=1, keepdims=True) + 1e-12)

        for q, e in zip(batch_qs, batch_embs):
            emb_map[q] = e

    return emb_map


def build_sorted_pools_from_emb_map(
    emb_map: Dict[str, np.ndarray],
    items: List[Tuple[str, List[str], Set[str]]]
) -> List[List[str]]:
    sorted_pools: List[List[str]] = []

    for anchor_q, pool, _ in items:
        a_emb = emb_map[anchor_q]
        pool_embs = np.vstack([emb_map[q] for q in pool])
        sims = pool_embs @ a_emb
        order = np.argsort(-sims)
        sorted_pools.append([pool[i] for i in order])

    return sorted_pools


# -------------------------------------------------------------------
# Binary metrics using annotated positives
# -------------------------------------------------------------------
def avg_prf_at_k_annotated(
    sorted_pools: List[List[str]],
    items: List[Tuple[str, List[str], Set[str]]],
    k: int
) -> Tuple[float, float, float]:
    p_sum = r_sum = f1_sum = 0.0
    total = 0

    for sorted_pool, (_, _, positives) in zip(sorted_pools, items):
        if not positives:
            continue

        topk = sorted_pool[:k]
        inter = len(set(topk) & positives)

        p = inter / k if k > 0 else 0.0
        r = inter / len(positives) if len(positives) > 0 else 0.0
        f1 = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0

        p_sum += p
        r_sum += r
        f1_sum += f1
        total += 1

    if total == 0:
        return 0.0, 0.0, 0.0

    return p_sum / total, r_sum / total, f1_sum / total


def avg_mrr_at_k(
    sorted_pools: List[List[str]],
    items: List[Tuple[str, List[str], Set[str]]],
    k: int
) -> float:
    rr_sum = 0.0
    total = 0

    for sorted_pool, (_, _, positives) in zip(sorted_pools, items):
        total += 1
        rr = 0.0
        for rank, q in enumerate(sorted_pool[:k], start=1):
            if q in positives:
                rr = 1.0 / rank
                break
        rr_sum += rr

    return rr_sum / max(total, 1)


def avg_hits_at_k(
    sorted_pools: List[List[str]],
    items: List[Tuple[str, List[str], Set[str]]],
    k: int
) -> float:
    hits = 0
    total = 0

    for sorted_pool, (_, _, positives) in zip(sorted_pools, items):
        total += 1
        if any(q in positives for q in sorted_pool[:k]):
            hits += 1

    return hits / max(total, 1)


# -------------------------------------------------------------------
# Optional graded metrics using Jaccard
# -------------------------------------------------------------------
def dcg(rels: List[float]) -> float:
    return float(sum((r / np.log2(i + 2)) for i, r in enumerate(rels)))


def avg_jaccard_at_k(sorted_pools: List[List[str]], k: int) -> float:
    total = 0.0
    count = 0

    for sorted_pool, rel_map in zip(sorted_pools, jaccard_scores_per_item):
        topk = [q for q in sorted_pool[:k] if q in rel_map]
        if not topk:
            continue
        total += float(np.mean([rel_map[q] for q in topk]))
        count += 1

    return total / max(count, 1)


def avg_ndcg_at_k(sorted_pools: List[List[str]], k: int) -> float:
    total = 0.0
    count = 0

    for sorted_pool, rel_map in zip(sorted_pools, jaccard_scores_per_item):
        topk = [q for q in sorted_pool[:k] if q in rel_map]
        if not topk:
            continue

        rels = [rel_map[q] for q in topk]
        dcg_val = dcg(rels)

        ideal_rels = sorted(rel_map.values(), reverse=True)[:k]
        idcg = dcg(ideal_rels)

        ndcg = (dcg_val / idcg) if idcg > 0 else 0.0
        total += ndcg
        count += 1

    return total / max(count, 1)


def avg_prf_at_k_jaccard_binary(sorted_pools: List[List[str]], k: int) -> Tuple[float, float, float]:
    p_sum = r_sum = f1_sum = 0.0
    total = 0

    for sorted_pool, rel_map in zip(sorted_pools, jaccard_scores_per_item):
        pos_set = {q for q, s in rel_map.items() if s >= JACCARD_BINARY_THRESHOLD}
        if not pos_set:
            continue

        topk_set = set(sorted_pool[:k])
        inter = len(topk_set & pos_set)

        p = inter / k if k > 0 else 0.0
        r = inter / len(pos_set) if len(pos_set) > 0 else 0.0
        f1 = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0

        p_sum += p
        r_sum += r
        f1_sum += f1
        total += 1

    if total == 0:
        return 0.0, 0.0, 0.0

    return p_sum / total, r_sum / total, f1_sum / total


# -------------------------------------------------------------------
# Reporting
# -------------------------------------------------------------------
def report(name: str, sorted_pools: List[List[str]]):
    print(f"\n{name}")
    print("Binary relevance (annotated positives):")
    for k in TOPKS:
        p, r, f1 = avg_prf_at_k_annotated(sorted_pools, eval_items_corrected, k)
        mrr = avg_mrr_at_k(sorted_pools, eval_items_corrected, k)
        hits = avg_hits_at_k(sorted_pools, eval_items_corrected, k)
        print(f"@{k:<2} | P: {p:.4f} | R: {r:.4f} | F1: {f1:.4f} | MRR: {mrr:.4f} | Hits: {hits:.4f}")

    print("\nOptional graded relevance (SPARQL answer Jaccard):")
    for k in TOPKS:
        aj = avg_jaccard_at_k(sorted_pools, k)
        nd = avg_ndcg_at_k(sorted_pools, k)
        print(f"@{k:<2} | avgJacc: {aj:.4f} | nDCG: {nd:.4f}")

    print(f"\nOptional binary relevance (Jaccard >= {JACCARD_BINARY_THRESHOLD}):")
    for k in TOPKS:
        p, r, f1 = avg_prf_at_k_jaccard_binary(sorted_pools, k)
        print(f"@{k:<2} | P: {p:.4f} | R: {r:.4f} | F1: {f1:.4f}")


# -------------------------------------------------------------------
# Encode all models
# -------------------------------------------------------------------
qwen = SentenceTransformer(QWEN_MODEL_NAME)
qwen_emb_map = build_emb_map(qwen, unique_queries, batch_size=BATCH)
qwen_sorted_pools = build_sorted_pools_from_emb_map(qwen_emb_map, eval_items_corrected)

minilm = SentenceTransformer(MINILM_MODEL_NAME)
minilm_emb_map = build_emb_map(minilm, unique_queries, batch_size=BATCH)
minilm_sorted_pools = build_sorted_pools_from_emb_map(minilm_emb_map, eval_items_corrected)

e5 = SentenceTransformer(E5_MODEL_NAME)
e5_emb_map = build_emb_map(
    e5,
    unique_queries,
    batch_size=BATCH,
    transform=lambda s: f"query: {s}"
)
e5_sorted_pools = build_sorted_pools_from_emb_map(e5_emb_map, eval_items_corrected)

mpnet = SentenceTransformer(MPNET_MODEL_NAME)
mpnet_emb_map = build_emb_map(mpnet, unique_queries, batch_size=BATCH)
mpnet_sorted_pools = build_sorted_pools_from_emb_map(mpnet_emb_map, eval_items_corrected)

tfidf = TfidfVectorizer(analyzer="word", ngram_range=(1, 2))
X = tfidf.fit_transform(unique_queries)
X = sk_normalize(X, norm="l2", axis=1)
q_to_row = {q: i for i, q in enumerate(unique_queries)}

tfidf_sorted_pools: List[List[str]] = []
for anchor_q, pool, _ in eval_items_corrected:
    a_row = q_to_row[anchor_q]
    a_vec = X[a_row]
    pool_rows = [q_to_row[q] for q in pool]
    pool_vecs = X[pool_rows]
    sims = (pool_vecs @ a_vec.T).toarray().ravel()
    order = np.argsort(-sims)
    tfidf_sorted_pools.append([pool[i] for i in order])

hybrid = SPARQLEmbedder(CHECKPOINT_PATH)
hybrid_emb_map = build_emb_map(
    hybrid,
    unique_queries,
    batch_size=BATCH,
    normalize_embeddings=True,
    use_convert_to_numpy=False
)
hybrid_sorted_pools = build_sorted_pools_from_emb_map(hybrid_emb_map, eval_items_corrected)


# -------------------------------------------------------------------
# Final reports
# -------------------------------------------------------------------
report("Baseline: Qwen", qwen_sorted_pools)
report("Baseline: MiniLM", minilm_sorted_pools)
report("Baseline: E5", e5_sorted_pools)
report("Baseline: MPNet", mpnet_sorted_pools)
report("Baseline: TF-IDF", tfidf_sorted_pools)
report("HybridSPARQLEncoder", hybrid_sorted_pools)

print("\nCached unique queries:", len(answer_cache))

Loaded 4713 semantic entries | Eval size: 472
Unique queries to execute/encode: 1881
Corrected eval items with non-empty annotated positives: 472
Executing SPARQL for unique queries (cached)...
Cached answers: 1881 | ok: 1881 | empty: 0 | failed: 0
Building graded Jaccard relevance maps...
Items with at least one valid graded Jaccard comparison: 472 / 472


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 440.29it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 432.65it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 458.29it/s, Materializing param=pooler.dense.weight

Loading model from ../checkpoints/dbpedia_model_semantic-20.pt...

Baseline: Qwen
Binary relevance (annotated positives):
@3  | P: 0.1949 | R: 0.2651 | F1: 0.1787 | MRR: 0.3771 | Hits: 0.4280
@5  | P: 0.1466 | R: 0.2831 | F1: 0.1475 | MRR: 0.3897 | Hits: 0.4831
@10 | P: 0.1017 | R: 0.3204 | F1: 0.1174 | MRR: 0.4012 | Hits: 0.5720
@20 | P: 0.0753 | R: 0.3785 | F1: 0.1003 | MRR: 0.4063 | Hits: 0.6462

Optional graded relevance (SPARQL answer Jaccard):
@3  | avgJacc: 0.2732 | nDCG: 0.4193
@5  | avgJacc: 0.2225 | nDCG: 0.4137
@10 | avgJacc: 0.1748 | nDCG: 0.4105
@20 | avgJacc: 0.1436 | nDCG: 0.4121

Optional binary relevance (Jaccard >= 0.8):
@3  | P: 0.2140 | R: 0.2656 | F1: 0.1798
@5  | P: 0.1627 | R: 0.2832 | F1: 0.1483
@10 | P: 0.1169 | R: 0.3206 | F1: 0.1199
@20 | P: 0.0867 | R: 0.3772 | F1: 0.1034

Baseline: MiniLM
Binary relevance (annotated positives):
@3  | P: 0.1850 | R: 0.2351 | F1: 0.1593 | MRR: 0.3492 | Hits: 0.4153
@5  | P: 0.1436 | R: 0.2665 | F1: 0.1405 | MRR: 0.3573 | Hits